In [ ]:
# In[1]:

import pandas as pd
import pytz

# Read CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# timezone
tz = pytz.timezone('Asia/Shanghai')

def ts_min_max_to_local(df, ts_col='timestamp'):
    # convert epoch seconds to timezone-aware datetimes in Asia/Shanghai
    if df.empty:
        return None, None
    ts_series = pd.to_datetime(df[ts_col], unit='s', utc=True).dt.tz_convert(tz)
    return ts_series.min().strftime('%Y-%m-%d %H:%M:%S %Z'), ts_series.max().strftime('%Y-%m-%d %H:%M:%S %Z')

def compact_unique_list(series, limit=20):
    if series is None:
        return ""
    unique_vals = pd.Series(series.dropna().unique()).astype(str)
    unique_vals = unique_vals.sort_values()
    top_vals = unique_vals.head(limit).tolist()
    return ", ".join(top_vals)

def top_pairs(df, left_col, right_col, topn=20):
    if df.empty:
        return pd.DataFrame(columns=[left_col, right_col, 'count'])
    grp = df.groupby([left_col, right_col]).size().reset_index(name='count')
    grp = grp.sort_values('count', ascending=False).head(topn).reset_index(drop=True)
    return grp

# metric.csv summary
metric_count = len(metric_df)
metric_min_ts, metric_max_ts = ts_min_max_to_local(metric_df, 'timestamp')
metric_unique_cmdb = compact_unique_list(metric_df['cmdb_id'], 20)
metric_unique_kpi_count = metric_df['kpi_name'].nunique()
metric_unique_kpi_list = compact_unique_list(metric_df['kpi_name'], 20)
metric_top20 = top_pairs(metric_df, 'cmdb_id', 'kpi_name', 20)

metric_summary = pd.DataFrame({
    'key': [
        'file','total_rows','min_timestamp','max_timestamp',
        'unique_cmdb_id_count','unique_cmdb_id_list',
        'unique_kpi_count','unique_kpi_list',
        'top_pairs_shown'
    ],
    'value': [
        'metric.csv', metric_count, metric_min_ts, metric_max_ts,
        metric_df['cmdb_id'].nunique(), metric_unique_cmdb,
        metric_unique_kpi_count, metric_unique_kpi_list,
        f"See metric_top20 ({metric_top20.shape[0]} rows)"
    ]
})

# trace.csv summary
trace_count = len(trace_df)
trace_min_ts, trace_max_ts = ts_min_max_to_local(trace_df, 'timestamp')
trace_unique_cmdb = compact_unique_list(trace_df['cmdb_id'], 20)
trace_unique_trace_count = trace_df['trace_name'].nunique()
trace_unique_trace_list = compact_unique_list(trace_df['trace_name'], 20)
trace_top20 = top_pairs(trace_df, 'cmdb_id', 'trace_name', 20)

trace_summary = pd.DataFrame({
    'key': [
        'file','total_rows','min_timestamp','max_timestamp',
        'unique_cmdb_id_count','unique_cmdb_id_list',
        'unique_trace_count','unique_trace_list',
        'top_pairs_shown'
    ],
    'value': [
        'trace.csv', trace_count, trace_min_ts, trace_max_ts,
        trace_df['cmdb_id'].nunique(), trace_unique_cmdb,
        trace_unique_trace_count, trace_unique_trace_list,
        f"See trace_top20 ({trace_top20.shape[0]} rows)"
    ]
})

# log.csv summary
log_count = len(log_df)
log_min_ts, log_max_ts = ts_min_max_to_local(log_df, 'timestamp')
log_unique_cmdb = compact_unique_list(log_df['cmdb_id'], 20)
log_unique_log_count = log_df['log_name'].nunique()
log_unique_log_list = compact_unique_list(log_df['log_name'], 20)
log_top20 = top_pairs(log_df, 'cmdb_id', 'log_name', 20)

log_summary = pd.DataFrame({
    'key': [
        'file','total_rows','min_timestamp','max_timestamp',
        'unique_cmdb_id_count','unique_cmdb_id_list',
        'unique_log_count','unique_log_list',
        'top_pairs_shown'
    ],
    'value': [
        'log.csv', log_count, log_min_ts, log_max_ts,
        log_df['cmdb_id'].nunique(), log_unique_cmdb,
        log_unique_log_count, log_unique_log_list,
        f"See log_top20 ({log_top20.shape[0]} rows)"
    ]
})

# error_logs.csv summary (top messages instead of KPI names)
error_count = len(error_df)
error_min_ts, error_max_ts = ts_min_max_to_local(error_df, 'timestamp')
error_unique_cmdb = compact_unique_list(error_df['cmdb_id'], 20)
error_unique_msg_count = error_df['message'].nunique() if 'message' in error_df.columns else 0
error_unique_msg_list = compact_unique_list(error_df['message'] if 'message' in error_df.columns else pd.Series(dtype=str), 20)

if 'message' in error_df.columns:
    error_top20 = error_df.groupby('message').size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
else:
    error_top20 = pd.DataFrame(columns=['message','count'])

error_summary = pd.DataFrame({
    'key': [
        'file','total_rows','min_timestamp','max_timestamp',
        'unique_cmdb_id_count','unique_cmdb_id_list',
        'unique_message_count','unique_message_list',
        'top_messages_shown'
    ],
    'value': [
        'error_logs.csv', error_count, error_min_ts, error_max_ts,
        error_df['cmdb_id'].nunique(), error_unique_cmdb,
        error_unique_msg_count, error_unique_msg_list,
        f"See error_top20 ({error_top20.shape[0]} rows)"
    ]
})

# Keep outputs compact: show summaries and top-20 tables (selected columns)
metric_summary, metric_top20.head(20), trace_summary, trace_top20.head(20), log_summary, log_top20.head(20), error_summary, error_top20.head(20)

```
Out[1]:
```
import pandas as pd

# Reuse existing summary DataFrames from the prior execution
ms = metric_summary.set_index('key')['value'].to_dict()
ts = trace_summary.set_index('key')['value'].to_dict()
ls = log_summary.set_index('key')['value'].to_dict()
es = error_summary.set_index('key')['value'].to_dict()

# Pull a couple of top error messages for emphasis
top_errors = error_top20.head(3).copy()
top_errors['message'] = top_errors['message'].astype(str)
top_err_lines = [f"{i+1}. {row['message']} (count={int(row['count'])})" 
                 for i, row in top_errors.iterrows()]

summary = (
    "Telemetry summary (compact):\n\n"
    "metric.csv:\n"
    f"- rows: {ms.get('total_rows')}; time range: {ms.get('min_timestamp')} to {ms.get('max_timestamp')}\n"
    f"- unique cmdb_id count: {ms.get('unique_cmdb_id_count')}; unique KPI count: {ms.get('unique_kpi_count')}\n"
    "- top metric producers include many KPIs from 'adservice' (multiple KPI series ~30 samples each) and a notable series on 'shippingservice2-0'.\n\n"
    "trace.csv:\n"
    f"- rows: {ts.get('total_rows')}; time range: {ts.get('min_timestamp')} to {ts.get('max_timestamp')}\n"
    f"- unique cmdb_id count: {ts.get('unique_cmdb_id_count')}; unique trace count: {ts.get('unique_trace_count')}\n"
    "- top trace pairs show heavy activity from adservice pods and shippingservice pods (many trace series ~30 samples each).\n\n"
    "log.csv:\n"
    f"- rows: {ls.get('total_rows')}; time range: {ls.get('min_timestamp')} to {ls.get('max_timestamp')}\n"
    f"- unique cmdb_id count: {ls.get('unique_cmdb_id_count')}; unique log types: {ls.get('unique_log_count')} ({ls.get('unique_log_list')})\n"
    "- logs are present across many service pods (e.g., adservice-*, cartservice-*, checkoutservice-*) with both log.error_count and log.row_count appearing frequently.\n\n"
    "error_logs.csv:\n"
    f"- rows: {es.get('total_rows')}; time range: {es.get('min_timestamp')} to {es.get('max_timestamp')}\n"
    f"- unique cmdb_id count: {es.get('unique_cmdb_id_count')} ({es.get('unique_cmdb_id_list')}); unique messages: {es.get('unique_message_count')}\n"
    "- top messages (top 3 shown):\n"
    + "\n".join(["  " + line for line in top_err_lines])
    + "\n\nOverall takeaway: most telemetry is concentrated in a 30-minute window (2022-03-21 11:00–11:29 CST). Metrics and traces show many repeated series from adservice and some shippingservice pods. Error logs are dominated by a single warning message from frontend pods (~1616 of 1708 entries), with a smaller set of error messages (≈47) and a few distinct HTTP 500 occurrences."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    key                                              value
0                  file                                         metric.csv
1            total_rows                                             148332
2         min_timestamp                            2022-03-21 11:00:00 CST
3         max_timestamp                            2022-03-21 11:29:00 CST
4  unique_cmdb_id_count                                                 59
5   unique_cmdb_id_list  adservice, adservice-0, adservice-1, adservice...
6      unique_kpi_count                                               1228
7       unique_kpi_list  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
8       top_pairs_shown                         See metric_top20 (20 rows),                cmdb_id                                           kpi_name  count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30
1            adservice                                     app.grpc.count     30
2            adservice                                       app.grpc.mrt     30
3            adservice                                        app.grpc.rr     30
4            adservice                                        app.grpc.sr     30
5            adservice                                     app.http.count     30
6            adservice                                       app.http.mrt     30
7            adservice                                        app.http.rr     30
8            adservice                                        app.http.sr     30
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30
12           adservice             runtime.java_lang_ClassLoading_Verbose     30
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...     30
14           adservice  runtime.java_lang_Compilation_TotalCompilation...     30
15           adservice  runtime.java_lang_GarbageCollector_CollectionC...     30
16           adservice  runtime.java_lang_GarbageCollector_CollectionC...     30
17           adservice  runtime.java_lang_GarbageCollector_CollectionT...     30
18           adservice  runtime.java_lang_GarbageCollector_CollectionT...     30
19           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     30,                     key                                              value
0                  file                                          trace.csv
1            total_rows                                              34304
2         min_timestamp                            2022-03-21 11:00:00 CST
3         max_timestamp                            2022-03-21 11:29:00 CST
4  unique_cmdb_id_count                                                 40
5   unique_cmdb_id_list  adservice-0, adservice-1, adservice-2, adservi...
6    unique_trace_count                                                200
7     unique_trace_list  trace.from_checkoutservice-0.duration_mean, tr...
8       top_pairs_shown                          See trace_top20 (20 rows),                cmdb_id                           trace_name  count
0   shippingservice2-0     trace.from_frontend2-0.row_count     30
1          adservice-0  trace.from_frontend-0.duration_mean     30
2          adservice-0   trace.from_frontend-0.duration_p95     30
3          adservice-0     trace.from_frontend-0.error_rate     30
4          adservice-0      trace.from_frontend-0.row_count     30
5          adservice-0  trace.from_frontend-1.duration_mean     30
6          adservice-0   trace.from_frontend-1.duration_p95     30
7          adservice-0     trace.from_frontend-1.error_rate     30
8          adservice-0      trace.from_frontend-1.row_count     30
9          adservice-0  trace.from_frontend-2.duration_mean     30
10         adservice-0   trace.from_frontend-2.duration_p95     30
11         adservice-0     trace.from_frontend-2.error_rate     30
12         adservice-0      trace.from_frontend-2.row_count     30
13         adservice-1  trace.from_frontend-0.duration_mean     30
14         adservice-1   trace.from_frontend-0.duration_p95     30
15         adservice-1     trace.from_frontend-0.error_rate     30
16   shippingservice-1  trace.from_frontend-1.duration_mean     30
17   shippingservice-1   trace.from_frontend-1.duration_p95     30
18   shippingservice-1     trace.from_frontend-1.error_rate     30
19   shippingservice-1      trace.from_frontend-1.row_count     30,                     key                                              value
0                  file                                            log.csv
1            total_rows                                               1758
2         min_timestamp                            2022-03-21 11:00:00 CST
3         max_timestamp                            2022-03-21 11:29:00 CST
4  unique_cmdb_id_count                                                 31
5   unique_cmdb_id_list  adservice-0, adservice-1, adservice-2, cartser...
6      unique_log_count                                                  2
7       unique_log_list                     log.error_count, log.row_count
8       top_pairs_shown                            See log_top20 (20 rows),               cmdb_id         log_name  count
0         adservice-0  log.error_count     30
1         adservice-0    log.row_count     30
2         adservice-1  log.error_count     30
3         adservice-1    log.row_count     30
4         adservice-2  log.error_count     30
5         adservice-2    log.row_count     30
6       cartservice-0  log.error_count     30
7       cartservice-0    log.row_count     30
8       cartservice-1  log.error_count     30
9       cartservice-1    log.row_count     30
10      cartservice-2  log.error_count     30
11      cartservice-2    log.row_count     30
12  checkoutservice-0  log.error_count     30
13  checkoutservice-0    log.row_count     30
14  checkoutservice-1  log.error_count     30
15  checkoutservice-1    log.row_count     30
16  checkoutservice-2  log.error_count     30
17  checkoutservice-2    log.row_count     30
18  currencyservice-0  log.error_count     30
19  currencyservice-0    log.row_count     30,                     key                                              value
0                  file                                     error_logs.csv
1            total_rows                                               1708
2         min_timestamp                            2022-03-21 11:00:00 CST
3         max_timestamp                            2022-03-21 11:29:58 CST
4  unique_cmdb_id_count                                                  3
5   unique_cmdb_id_list                 frontend-0, frontend-1, frontend-2
6  unique_message_count                                                 47
7   unique_message_list  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...
8    top_messages_shown                          See error_top20 (20 rows),                                               message  count
0   severity: warning, message: failed to retrieve...   1616
1             severity: error, message: request error     47
2   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
3   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
4   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
5   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
6   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
7   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
8   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
9   "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
10  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
11  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
12  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
13  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
14  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
15  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
16  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
17  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
18  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1
19  "POST /cart/checkout HTTP/1.1" 500 - via_upstr...      1)```
```

In [ ]:
# In[2]:

# reuse metric_df and log_df from the existing kernel state
import pandas as pd

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_groups = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    p95=('value', lambda x: x.quantile(0.95)),
    min=('value', 'min'),
    max=('value', 'max')
).reset_index()

metric_top20 = metric_groups.sort_values('count', ascending=False).head(20)
# Ensure column order
metric_top20 = metric_top20[['cmdb_id', 'kpi_name', 'count', 'p95', 'min', 'max']]

# Aggregate log.csv by (cmdb_id, log_name)
log_groups = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    p95=('value', lambda x: x.quantile(0.95)),
    min=('value', 'min'),
    max=('value', 'max')
).reset_index()

log_top20 = log_groups.sort_values('count', ascending=False).head(20)
log_top20 = log_top20[['cmdb_id', 'log_name', 'count', 'p95', 'min', 'max']]

# Display compact results (up to 20 rows each)
metric_top20, log_top20

```
Out[2]:
```
summary = (
    "Summary of grouped results (top 20 by count):\n\n"
    "Metrics (metric.csv):\n"
    "- Top groups all have count = 30 (30 samples each).\n"
    "- The single most prominent metric group is shippingservice2-0 -> mesh.source... (count=30) with p95=2022.34, min=497.50, max=2139.25 — a high and wide-ranging series that stands out.\n"
    "- Many top groups are from adservice (count=30) including app.grpc.count (p95=238, min=152, max=243), app.grpc.mrt (p95≈0.00284), app.http.count (p95=24), and many JVM/runtime metrics (several constant-valued series like LoadedClassCount=4990).\n\n"
    "Logs (log.csv):\n"
    "- Top log groups also have count = 30 and are dominated by service pods (e.g., adservice-0/1/2, cartservice-*, checkoutservice-*).\n"
    "- log.row_count shows substantial traffic for some pods (example: adservice-0 row_count p95=1363, cartservice pods p95≈935), while log.error_count has p95=0 for the top pods (indicating few errors at the 95th percentile).\n\n"
    "Takeaway: the dataset’s top groups are uniformly sampled (30 points each). Shippingservice2-0’s mesh metric is an outlier in magnitude and range and may merit investigation; adservice emits many active application and JVM metrics. Logs indicate high request/row counts on several pods but low error counts at p95."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count           p95           min           max
4957  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30   2022.337500    497.500000   2139.250000
0              adservice                                     app.grpc.count     30    238.000000    152.000000    243.000000
1              adservice                                       app.grpc.mrt     30      0.002839      0.002406      0.003469
2              adservice                                        app.grpc.rr     30    100.000000    100.000000    100.000000
3              adservice                                        app.grpc.sr     30     77.412921     72.839506     78.289474
4              adservice                                     app.http.count     30     24.000000     24.000000     24.000000
5              adservice                                       app.http.mrt     30      0.009604      0.009163      0.009840
6              adservice                                        app.http.rr     30    100.000000    100.000000    100.000000
7              adservice                                        app.http.sr     30    100.000000    100.000000    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30   4990.000000   4990.000000   4990.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30   4990.000000   4990.000000   4990.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30      0.000000      0.000000      0.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     30      0.000000      0.000000      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     30      1.000000      1.000000      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...     30  43884.000000  43861.000000  43884.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30  45728.062500  45483.500000  45741.000000
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...     30      3.000000      3.000000      3.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30  71842.820833  71456.666667  71863.250000
17             adservice  runtime.java_lang_GarbageCollector_CollectionT...     30    177.000000    177.000000    177.000000
18             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     30      1.000000      1.000000      1.000000,               cmdb_id         log_name  count      p95    min     max
0         adservice-0  log.error_count     30     0.00    0.0     0.0
1         adservice-0    log.row_count     30  1363.00  834.0  1390.0
2         adservice-1  log.error_count     30     0.00    0.0     0.0
3         adservice-1    log.row_count     30   123.10   78.0   128.0
4         adservice-2  log.error_count     30     0.00    0.0     0.0
5         adservice-2    log.row_count     30   120.00   84.0   122.0
6       cartservice-0  log.error_count     30     0.00    0.0     0.0
7       cartservice-0    log.row_count     30   935.30  607.0   939.0
8       cartservice-1  log.error_count     30     0.00    0.0     0.0
9       cartservice-1    log.row_count     30   935.85  639.0   958.0
10      cartservice-2  log.error_count     30     0.00    0.0     0.0
11      cartservice-2    log.row_count     30   939.10  590.0   957.0
12  checkoutservice-0  log.error_count     30     0.00    0.0     0.0
13  checkoutservice-0    log.row_count     30    84.55    7.0    87.0
14  checkoutservice-1  log.error_count     30     0.00    0.0     0.0
15  checkoutservice-1    log.row_count     30    84.00    6.0    85.0
16  checkoutservice-2  log.error_count     30     0.00    0.0     0.0
17  checkoutservice-2    log.row_count     30    83.10    6.0    98.0
18  currencyservice-0  log.error_count     30     0.00    0.0     0.0
19  currencyservice-0    log.row_count     30   833.15  529.0   873.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import timedelta

# reuse metric_df from kernel state
tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is integer
metric_df['timestamp'] = metric_df['timestamp'].astype(int)

# 1) Calculate global P95 per (cmdb_id, kpi_name) using the entire series (before any filtering)
global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# Merge global p95 back to metric_df
metric_with_p95 = metric_df.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# 2) Define incident window in UTC based on Asia/Shanghai times (inclusive)
start_local = pd.Timestamp("2022-03-21 11:00:00", tz=tz)
end_local   = pd.Timestamp("2022-03-21 11:30:00", tz=tz)
start_utc_s = int(start_local.tz_convert('UTC').timestamp())
end_utc_s   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter rows to the incident window (inclusive)
window_df = metric_with_p95[(metric_with_p95['timestamp'] >= start_utc_s) & (metric_with_p95['timestamp'] <= end_utc_s)].copy()

# 4) Mark anomalies in the window where value > global p95
window_df['is_anom'] = window_df['value'] > window_df['p95']

# 5) For each group with at least one anomalous point compute required metrics
def compute_group_anomaly_info(g):
    anom_rows = g[g['is_anom']].sort_values('timestamp')
    if anom_rows.empty:
        return None
    anomaly_count = int(len(anom_rows))
    earliest_ts = int(anom_rows['timestamp'].min())
    # convert earliest timestamp to Asia/Shanghai tz string
    earliest_ts_local = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    # compute max consecutive anomalies (consecutive minutes -> diff == 60)
    ts_sorted = anom_rows['timestamp'].sort_values().to_numpy()
    max_run = 1
    current_run = 1
    if len(ts_sorted) == 1:
        max_run = 1
    else:
        prev = ts_sorted[0]
        for cur in ts_sorted[1:]:
            if cur - prev == 60:
                current_run += 1
            else:
                if current_run > max_run:
                    max_run = current_run
                current_run = 1
            prev = cur
        if current_run > max_run:
            max_run = current_run
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts_local,
        'max_consecutive_anomalies': int(max_run)
    })

grouped = window_df.groupby(['cmdb_id','kpi_name'])
anomaly_info = grouped.apply(compute_group_anomaly_info).dropna().reset_index()

# 6) Keep only groups that had >=1 anomaly and order by anomaly_count desc, limit to top 20
anomaly_summary_top20 = anomaly_info.sort_values('anomaly_count', ascending=False).head(20)[
    ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_consecutive_anomalies']
]

# Display compact result
anomaly_summary_top20

```
Out[3]:
```
```python
# Summarize anomaly analysis results in plain English (compact)
# Reuse anomaly_summary_top20 from previous kernel state

n_groups = int(anomaly_summary_top20.shape[0])
top_components = anomaly_summary_top20['cmdb_id'].value_counts().head(5).to_dict()
earliest_overall = anomaly_summary_top20['earliest_anomaly_timestamp'].min()
latest_examples = anomaly_summary_top20.sort_values('earliest_anomaly_timestamp', ascending=False).head(3)

example_lines = []
for _, r in latest_examples.iterrows():
    example_lines.append(f"{r['cmdb_id']} / {r['kpi_name']} (anomalies={int(r['anomaly_count'])}, first={r['earliest_anomaly_timestamp']}, max_run={int(r['max_consecutive_anomalies'])})")

max_run_overall = int(anomaly_summary_top20['max_consecutive_anomalies'].max())

summary = (
    f"Anomaly summary (window 2022-03-21 11:00–11:30 CST):\n"
    f"- Groups with anomalies found: {n_groups} (top 20 shown).\n"
    f"- Affected components (top by group count): {', '.join([f'{k}({v})' for k,v in top_components.items()])}.\n"
    f"- All reported groups have small anomaly counts (mostly 2 points each).\n"
    f"- Earliest anomalous timestamp among these groups: {earliest_overall}.\n"
    f"- Longest consecutive anomaly run observed: {max_run_overall} minute(s).\n"
    f"- Examples (most recent):\n  - " + "\n  - ".join(example_lines) + "\n\n"
    f"Takeaway: anomalies are short-lived and sparse (typically 2 minutes), concentrated in adservice metrics and a mesh metric on shippingservice2-0. Investigate those components around the listed timestamps (e.g., 11:04, 11:16, 11:27–11:28 CST) for root cause clues."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_consecutive_anomalies
1751  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....            2.0    2022-03-21 11:27:00 CST                        1.0
1750  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....            2.0    2022-03-21 11:27:00 CST                        1.0
1              adservice                                       app.grpc.mrt            2.0    2022-03-21 11:02:00 CST                        1.0
2              adservice                                        app.grpc.sr            2.0    2022-03-21 11:07:00 CST                        1.0
3              adservice                                       app.http.mrt            2.0    2022-03-21 11:04:00 CST                        1.0
4              adservice  runtime.java_lang_GarbageCollector_CollectionC...            2.0    2022-03-21 11:28:00 CST                        2.0
5              adservice  runtime.java_lang_GarbageCollector_CollectionT...            2.0    2022-03-21 11:28:00 CST                        2.0
6              adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:11:00 CST                        1.0
7              adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:28:00 CST                        2.0
8              adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:28:00 CST                        2.0
9              adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:04:00 CST                        1.0
10             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:28:00 CST                        2.0
11             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:16:00 CST                        1.0
12             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:28:00 CST                        2.0
13             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            2.0    2022-03-21 11:28:00 CST                        2.0
14             adservice  runtime.java_lang_MemoryPool_CollectionUsage_u...            2.0    2022-03-21 11:04:00 CST                        1.0
15             adservice  runtime.java_lang_MemoryPool_PeakUsage_used.Te...            2.0    2022-03-21 11:28:00 CST                        2.0
16             adservice  runtime.java_lang_MemoryPool_Usage_used.Eden_S...            2.0    2022-03-21 11:16:00 CST                        1.0
17             adservice  runtime.java_lang_MemoryPool_Usage_used.Surviv...            2.0    2022-03-21 11:04:00 CST                        1.0
18             adservice  runtime.java_lang_MemoryPool_Usage_used.Tenure...            2.0    2022-03-21 11:28:00 CST                        2.0```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is integer
metric_df['timestamp'] = metric_df['timestamp'].astype(int)

# 1) Global P95 per (cmdb_id, kpi_name) computed from the full series (no filtering)
global_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# Merge P95 back to full metric dataframe
metric_with_p95 = metric_df.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# 2) Define incident window (Asia/Shanghai timezone) and convert to UTC epoch seconds
start_local = pd.Timestamp("2022-03-21 11:00:00", tz=tz)
end_local   = pd.Timestamp("2022-03-21 11:30:00", tz=tz)
start_utc_s = int(start_local.tz_convert('UTC').timestamp())
end_utc_s   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter metric rows to the incident window (inclusive)
window_df = metric_with_p95[(metric_with_p95['timestamp'] >= start_utc_s) & (metric_with_p95['timestamp'] <= end_utc_s)].copy()

# 4) Mark anomaly points where value > group's global P95
window_df['is_anom'] = window_df['value'] > window_df['p95']

# 5) For each group with any anomaly, find earliest anomaly timestamp (min timestamp where is_anom is True)
anom_groups = (
    window_df[window_df['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(earliest_ts=('timestamp','min'))
)

# Convert earliest_ts to Asia/Shanghai localized string
if not anom_groups.empty:
    anom_groups['earliest_anomaly_timestamp'] = pd.to_datetime(anom_groups['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    anom_groups = anom_groups.drop(columns=['earliest_ts'])
else:
    # empty structure
    anom_groups['earliest_anomaly_timestamp'] = pd.Series(dtype=str)

# 1) Table: counts of groups by earliest_anomaly_timestamp, ordered by group_count desc (top 20)
earliest_counts = (
    anom_groups
    .groupby('earliest_anomaly_timestamp', as_index=False)
    .size()
    .rename(columns={'size':'group_count'})
    .sort_values('group_count', ascending=False)
    .head(20)
)

# 2) For the top 2 earliest_anomaly_timestamp values, breakdown of affected cmdb_id counts
top_ts = earliest_counts['earliest_anomaly_timestamp'].head(2).tolist()

# Prepare per-timestamp breakdowns (top 20 cmdb_id rows per timestamp)
breakdown_ts1 = pd.DataFrame(columns=['earliest_anomaly_timestamp','cmdb_id','groups_at_timestamp'])
breakdown_ts2 = pd.DataFrame(columns=['earliest_anomaly_timestamp','cmdb_id','groups_at_timestamp'])

if len(top_ts) >= 1:
    df1 = anom_groups[anom_groups['earliest_anomaly_timestamp'] == top_ts[0]]
    breakdown_ts1 = (
        df1.groupby(['earliest_anomaly_timestamp','cmdb_id'], as_index=False)
        .size()
        .rename(columns={'size':'groups_at_timestamp'})
        .sort_values('groups_at_timestamp', ascending=False)
        .head(20)
    )

if len(top_ts) >= 2:
    df2 = anom_groups[anom_groups['earliest_anomaly_timestamp'] == top_ts[1]]
    breakdown_ts2 = (
        df2.groupby(['earliest_anomaly_timestamp','cmdb_id'], as_index=False)
        .size()
        .rename(columns={'size':'groups_at_timestamp'})
        .sort_values('groups_at_timestamp', ascending=False)
        .head(20)
    )

# Display compact results: earliest_counts (<=20 rows) and up to two breakdown tables (each <=20 rows)
earliest_counts, breakdown_ts1, breakdown_ts2

```
Out[4]:
```
Summary of anomaly timing and affected components (window 2022-03-21 11:00–11:30 CST):

1) Groups by earliest anomaly timestamp (top counts)
- 2022-03-21 11:28:00 CST — 160 groups (most)
- 2022-03-21 11:11:00 CST — 148 groups
- 2022-03-21 11:10:00 CST — 136 groups
- 2022-03-21 11:02:00 CST — 129 groups
- 2022-03-21 11:15:00 CST — 126 groups
(Other timestamps with notable counts: 11:13:00=90, 11:20:00=68, 11:26:00=67, 11:00:00=59, 11:05/11:03=56 each.)

2) Breakdown for the top 2 timestamps

- For 2022-03-21 11:28:00 CST (top affected cmdb_ids, by number of metric groups anomalous at that timestamp):
  - adservice2: 21 groups
  - adservice: 17 groups
  - frontend-2: 13 groups
  - productcatalogservice2-0: 12 groups
  - cartservice-1: 8 groups
  - cartservice2-0: 6 groups
  - cartservice-2: 5 groups
  - cartservice-0: 5 groups
  - checkoutservice-1: 4 groups
  - productcatalogservice-2: 4 groups
  - productcatalogservice-1: 4 groups
  - (others with smaller counts include currencyservice2-0, paymentservice-1, frontend2-0, node-2, shippingservice-0, recommendationservice-1/2, adservice-0, checkoutservice-2, etc.)

- For 2022-03-21 11:11:00 CST (top affected cmdb_ids):
  - checkoutservice-2: 27 groups
  - currencyservice-1: 10 groups
  - frontend-1: 10 groups
  - emailservice-0: 9 groups
  - productcatalogservice-2: 8 groups
  - shippingservice-1: 8 groups
  - cartservice-2: 7 groups
  - shippingservice-0: 6 groups
  - productcatalogservice-1: 6 groups
  - currencyservice-2: 6 groups
  - (others: adservice-0, cartservice-0, emailservice-1, paymentservice-2, frontend-2, paymentservice-0, node-3, adservice2-0, checkoutservice-0, currencyservice-0)

Takeaway / recommended focus
- Two clear peaks of group-level anomalies: 11:28 and 11:11 CST. Investigate components with the highest group counts at those times:
  - Around 11:28 — adservice/adservice2, frontend-2, productcatalogservice2-0, and several cart/checkout/productcatalog pods.
  - Around 11:11 — checkoutservice-2 (largest), currencyservice-1, frontend-1, emailservice-0, and related productcatalog/shipping pods.
- These service/pod clusters are the best starting points for root-cause investigation for the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(   earliest_anomaly_timestamp  group_count
28    2022-03-21 11:28:00 CST          160
11    2022-03-21 11:11:00 CST          148
10    2022-03-21 11:10:00 CST          136
2     2022-03-21 11:02:00 CST          129
15    2022-03-21 11:15:00 CST          126
13    2022-03-21 11:13:00 CST           90
20    2022-03-21 11:20:00 CST           68
26    2022-03-21 11:26:00 CST           67
0     2022-03-21 11:00:00 CST           59
5     2022-03-21 11:05:00 CST           56
3     2022-03-21 11:03:00 CST           56
21    2022-03-21 11:21:00 CST           54
27    2022-03-21 11:27:00 CST           53
16    2022-03-21 11:16:00 CST           53
18    2022-03-21 11:18:00 CST           50
19    2022-03-21 11:19:00 CST           49
4     2022-03-21 11:04:00 CST           44
6     2022-03-21 11:06:00 CST           39
9     2022-03-21 11:09:00 CST           38
12    2022-03-21 11:12:00 CST           37,    earliest_anomaly_timestamp                   cmdb_id  groups_at_timestamp
4     2022-03-21 11:28:00 CST                adservice2                   21
0     2022-03-21 11:28:00 CST                 adservice                   17
24    2022-03-21 11:28:00 CST                frontend-2                   13
36    2022-03-21 11:28:00 CST  productcatalogservice2-0                   12
7     2022-03-21 11:28:00 CST             cartservice-1                    8
9     2022-03-21 11:28:00 CST            cartservice2-0                    6
8     2022-03-21 11:28:00 CST             cartservice-2                    5
6     2022-03-21 11:28:00 CST             cartservice-0                    5
11    2022-03-21 11:28:00 CST         checkoutservice-1                    4
35    2022-03-21 11:28:00 CST   productcatalogservice-2                    4
34    2022-03-21 11:28:00 CST   productcatalogservice-1                    4
17    2022-03-21 11:28:00 CST        currencyservice2-0                    3
30    2022-03-21 11:28:00 CST          paymentservice-1                    3
25    2022-03-21 11:28:00 CST               frontend2-0                    3
27    2022-03-21 11:28:00 CST                    node-2                    3
41    2022-03-21 11:28:00 CST         shippingservice-0                    2
38    2022-03-21 11:28:00 CST   recommendationservice-1                    2
12    2022-03-21 11:28:00 CST         checkoutservice-2                    2
1     2022-03-21 11:28:00 CST               adservice-0                    2
39    2022-03-21 11:28:00 CST   recommendationservice-2                    2,    earliest_anomaly_timestamp                  cmdb_id  groups_at_timestamp
10    2022-03-21 11:11:00 CST        checkoutservice-2                   27
13    2022-03-21 11:11:00 CST        currencyservice-1                   10
18    2022-03-21 11:11:00 CST               frontend-1                   10
15    2022-03-21 11:11:00 CST           emailservice-0                    9
28    2022-03-21 11:11:00 CST  productcatalogservice-2                    8
34    2022-03-21 11:11:00 CST        shippingservice-1                    8
5     2022-03-21 11:11:00 CST            cartservice-2                    7
33    2022-03-21 11:11:00 CST        shippingservice-0                    6
27    2022-03-21 11:11:00 CST  productcatalogservice-1                    6
14    2022-03-21 11:11:00 CST        currencyservice-2                    6
1     2022-03-21 11:11:00 CST              adservice-0                    4
4     2022-03-21 11:11:00 CST            cartservice-0                    4
16    2022-03-21 11:11:00 CST           emailservice-1                    4
25    2022-03-21 11:11:00 CST         paymentservice-2                    4
19    2022-03-21 11:11:00 CST               frontend-2                    4
23    2022-03-21 11:11:00 CST         paymentservice-0                    3
22    2022-03-21 11:11:00 CST                   node-3                    3
3     2022-03-21 11:11:00 CST             adservice2-0                    3
8     2022-03-21 11:11:00 CST        checkoutservice-0                    3
12    2022-03-21 11:11:00 CST        currencyservice-0                    2)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is integer
trace_df['timestamp'] = trace_df['timestamp'].astype(int)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series
trace_p95 = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# Merge p95 back
trace_with_p95 = trace_df.merge(trace_p95, on=['cmdb_id','trace_name'], how='left')

# 2) Define incident window in Asia/Shanghai and convert to UTC epoch seconds (inclusive)
start_local = pd.Timestamp("2022-03-21 11:00:00", tz=tz)
end_local   = pd.Timestamp("2022-03-21 11:30:00", tz=tz)
start_utc_s = int(start_local.tz_convert('UTC').timestamp())
end_utc_s   = int(end_local.tz_convert('UTC').timestamp())

# 3) Filter to window
window_trace = trace_with_p95[(trace_with_p95['timestamp'] >= start_utc_s) & (trace_with_p95['timestamp'] <= end_utc_s)].copy()

# 4) Mark anomalies where value > group P95
window_trace['is_anom'] = window_trace['value'] > window_trace['p95']

# 5) For groups with any anomaly, compute anomaly_count and earliest anomaly timestamp
anom_groups = (
    window_trace[window_trace['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_ts=('timestamp','min'))
)

# Convert earliest_ts to Asia/Shanghai formatted string
if not anom_groups.empty:
    anom_groups['earliest_anomaly_timestamp'] = pd.to_datetime(anom_groups['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    anom_groups = anom_groups.drop(columns=['earliest_ts'])
else:
    # keep consistent empty frame with required columns
    anom_groups = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp'])

# 6) Select groups for the two requested timestamps and order by anomaly_count desc, limit 20
ts_a = "2022-03-21 11:11:00 CST"
ts_b = "2022-03-21 11:28:00 CST"

groups_11_11 = (anom_groups[anom_groups['earliest_anomaly_timestamp'] == ts_a]
                .sort_values('anomaly_count', ascending=False)
                .reset_index(drop=True)
                .head(20)
                [['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp']])

groups_11_28 = (anom_groups[anom_groups['earliest_anomaly_timestamp'] == ts_b]
                .sort_values('anomaly_count', ascending=False)
                .reset_index(drop=True)
                .head(20)
                [['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp']])

# Display the two compact tables
groups_11_11, groups_11_28

```
Out[5]:
```
summary = (
    "Trace anomaly summary (window 2022-03-21 11:00–11:30 CST):\n\n"
    "1) Timestamp 2022-03-21 11:11:00 CST:\n"
    "- Many trace groups (shown results include 20 representative groups) have earliest_anomaly_timestamp = 11:11 and anomaly_count = 2 (i.e., two anomalous minutes each).\n"
    "- Affected components include adservice-0, checkoutservice-2, frontend-0, frontend-2, shippingservice-0/1, productcatalogservice-1 and others.\n"
    "- Typical traces involved: frontend→adservice traces, checkout→shippingservice traces, and cross-service duration and p95 metrics — indicating a broad, multi-service latency/traffic spike around 11:11.\n\n"
    "2) Timestamp 2022-03-21 11:28:00 CST:\n"
    "- Only two trace groups were detected with earliest_anomaly_timestamp = 11:28, each with anomaly_count = 1.\n"
    "- These are: checkoutservice-2 → trace.to_currencyservice-1.row_count and currencyservice-1 → trace.from_checkoutservice-2.row_count — a small, localized anomaly between checkoutservice-2 and currencyservice-1.\n\n"
    "Takeaway / next steps:\n"
    "- Prioritize investigation around 11:11 CST: examine latency, request rates, and inter-service calls for frontend→adservice and checkout↔shipping paths (these show the strongest, multi-group anomalies).\n"
    "- Treat the 11:28 anomalies as secondary/localized (checkoutservice-2 <-> currencyservice-1) but check for any related retries or cascading effects.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                        trace_name  anomaly_count earliest_anomaly_timestamp
0               adservice-0               trace.from_frontend-0.duration_mean              2    2022-03-21 11:11:00 CST
1               adservice-0                trace.from_frontend-0.duration_p95              2    2022-03-21 11:11:00 CST
2               adservice-0                   trace.from_frontend-2.row_count              2    2022-03-21 11:11:00 CST
3               adservice-1                   trace.from_frontend-2.row_count              2    2022-03-21 11:11:00 CST
4         checkoutservice-2          trace.to_shippingservice-0.duration_mean              2    2022-03-21 11:11:00 CST
5         checkoutservice-1              trace.to_shippingservice-0.row_count              2    2022-03-21 11:11:00 CST
6         checkoutservice-2           trace.to_shippingservice-1.duration_p95              2    2022-03-21 11:11:00 CST
7                frontend-0                trace.to_adservice-0.duration_mean              2    2022-03-21 11:11:00 CST
8         checkoutservice-2          trace.to_shippingservice-1.duration_mean              2    2022-03-21 11:11:00 CST
9         checkoutservice-2           trace.to_shippingservice-0.duration_p95              2    2022-03-21 11:11:00 CST
10  productcatalogservice-1   trace.from_recommendationservice-1.duration_p95              2    2022-03-21 11:11:00 CST
11               frontend-2                    trace.to_adservice-1.row_count              2    2022-03-21 11:11:00 CST
12               frontend-2                    trace.to_adservice-0.row_count              2    2022-03-21 11:11:00 CST
13               frontend-0                 trace.to_adservice-0.duration_p95              2    2022-03-21 11:11:00 CST
14        shippingservice-0         trace.from_checkoutservice-2.duration_p95              2    2022-03-21 11:11:00 CST
15        shippingservice-1        trace.from_checkoutservice-2.duration_mean              2    2022-03-21 11:11:00 CST
16        shippingservice-1         trace.from_checkoutservice-2.duration_p95              2    2022-03-21 11:11:00 CST
17        shippingservice-0        trace.from_checkoutservice-2.duration_mean              2    2022-03-21 11:11:00 CST
18  productcatalogservice-1  trace.from_recommendationservice-1.duration_mean              2    2022-03-21 11:11:00 CST
19        shippingservice-0            trace.from_checkoutservice-1.row_count              2    2022-03-21 11:11:00 CST,              cmdb_id                              trace_name  anomaly_count earliest_anomaly_timestamp
0  checkoutservice-2    trace.to_currencyservice-1.row_count              1    2022-03-21 11:28:00 CST
1  currencyservice-1  trace.from_checkoutservice-2.row_count              1    2022-03-21 11:28:00 CST)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing dataframes if present, else load
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp type
error_df['timestamp'] = error_df['timestamp'].astype(int)
log_df['timestamp'] = log_df['timestamp'].astype(int)

# Incident window in Asia/Shanghai (inclusive) -> convert to UTC epoch seconds
start_local = pd.Timestamp("2022-03-21 11:00:00", tz=tz)
end_local   = pd.Timestamp("2022-03-21 11:30:00", tz=tz)
start_utc_s = int(start_local.tz_convert('UTC').timestamp())
end_utc_s   = int(end_local.tz_convert('UTC').timestamp())

# 1) error_logs: filter to window
window_error = error_df[(error_df['timestamp'] >= start_utc_s) & (error_df['timestamp'] <= end_utc_s)].copy()

# Total count per minute (timestamp)
per_min_total = window_error.groupby('timestamp').size().reset_index(name='total_count')

# Count per (timestamp, cmdb_id)
per_min_cmdb = window_error.groupby(['timestamp','cmdb_id']).size().reset_index(name='cmdb_count')

# For each timestamp, find top cmdb_id and its count
per_min_cmdb_sorted = per_min_cmdb.sort_values(['timestamp','cmdb_count'], ascending=[True,False])
top_cmdb_per_min = per_min_cmdb_sorted.groupby('timestamp', as_index=False).first()[['timestamp','cmdb_id','cmdb_count']].rename(columns={'cmdb_id':'top_cmdb_id','cmdb_count':'top_cmdb_id_count'})

# Merge totals with top cmdbs
top_minutes = per_min_total.merge(top_cmdb_per_min, on='timestamp', how='left')

# Convert timestamp to Asia/Shanghai localized string for display
top_minutes['timestamp'] = pd.to_datetime(top_minutes['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select top 10 minutes by total_count desc
top_minutes_table = top_minutes.sort_values('total_count', ascending=False).head(10)[['timestamp','total_count','top_cmdb_id','top_cmdb_id_count']]

# 2) log.csv: filter to window and log.error_count > 0
window_log = log_df[(log_df['timestamp'] >= start_utc_s) & (log_df['timestamp'] <= end_utc_s)].copy()
error_count_nonzero = window_log[(window_log['log_name'] == 'log.error_count') & (window_log['value'] > 0)].copy()

# (a) up to top 20 rows ordered by value desc
if not error_count_nonzero.empty:
    error_count_nonzero['timestamp'] = pd.to_datetime(error_count_nonzero['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    top20_nonzero_logs = error_count_nonzero.sort_values('value', ascending=False).head(20)[['timestamp','cmdb_id','log_name','value']]
else:
    top20_nonzero_logs = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

# (b) aggregate counts per cmdb_id and earliest timestamp when value>0
if not error_count_nonzero.empty:
    agg_nonzero = (
        error_count_nonzero
        .groupby('cmdb_id', as_index=False)
        .agg(nonzero_count=('value','count'), earliest_ts=('timestamp', 'min'))
    )
    # earliest_ts currently formatted as string? If not, ensure conversion:
    # If earliest_ts is numeric epoch (it was converted earlier to str), handle both:
    def fmt_ts(x):
        try:
            # if numeric (int)
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            # assume already formatted string
            return str(x)
    # If earliest_ts are strings of timestamps already, they are in local string; but we converted timestamp column to strings earlier,
    # so earliest_ts may be those strings. To be safe, recompute earliest_ts from original filtered numeric df:
    agg_nonzero = (
        error_count_nonzero.assign(ts_numeric=pd.to_datetime(error_count_nonzero['timestamp'], utc=True).astype('int64')//10**9)
        .groupby('cmdb_id', as_index=False)
        .agg(nonzero_count=('value','count'), earliest_ts_numeric=('ts_numeric','min'))
    )
    agg_nonzero['earliest_nonzero_timestamp'] = pd.to_datetime(agg_nonzero['earliest_ts_numeric'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    agg_nonzero_per_cmdb = agg_nonzero[['cmdb_id','nonzero_count','earliest_nonzero_timestamp']].sort_values('nonzero_count', ascending=False).head(20)
else:
    agg_nonzero_per_cmdb = pd.DataFrame(columns=['cmdb_id','nonzero_count','earliest_nonzero_timestamp'])

# Final compact outputs
top_minutes_table, top20_nonzero_logs, agg_nonzero_per_cmdb

```
Out[6]:
```
```python
summary = (
    "Error log analysis (window 2022-03-21 11:00–11:30 CST):\n\n"
    "1) Top minutes by error_log count (top 10 shown):\n"
    "- 2022-03-21 11:07:08 CST — total_count=6; top_cmdb_id=frontend-0 (4 msgs)\n"
    "- 2022-03-21 11:09:00 CST — total_count=5; top_cmdb_id=frontend-1 (3 msgs)\n"
    "- Several other minutes have 4 messages each (examples at 11:07:28, 11:09:08, 11:09:02, 11:05:18, 11:25:03, 11:05:57, 11:04:44, 11:08:18) often dominated by frontend pods.\n\n"
    "2) log.csv (log.error_count > 0) within the same window:\n"
    "a) Top per-minute nonzero entries (up to 20 shown, highest values):\n"
    "- 2022-03-21 11:09:00 CST, frontend-2, log.error_count = 29.0\n"
    "- 2022-03-21 11:07:00 CST, frontend-2, log.error_count = 27.0\n"
    "- 2022-03-21 11:05:00 CST, frontend-0, log.error_count = 26.0\n"
    "- 2022-03-21 11:09:00 CST, frontend-1, log.error_count = 25.0\n"
    "- Additional high values appear across frontend-2 and frontend-1 at various minutes (many entries in the 20–29 range).\n\n"
    "b) Aggregate of non-zero log.error_count occurrences per component (top results):\n"
    "- frontend-0: nonzero_count = 30, earliest_nonzero_timestamp = 2022-03-21 19:00:00 CST\n"
    "- frontend-1: nonzero_count = 30, earliest_nonzero_timestamp = 2022-03-21 19:00:00 CST\n"
    "- frontend-2: nonzero_count = 30, earliest_nonzero_timestamp = 2022-03-21 19:00:00 CST\n\n"
    "Notes / takeaway:\n"
    "- Error messages are concentrated in short spikes (per-minute peaks) and are dominated by frontend pods.\n"
    "- Frontend pods show sustained non-zero log.error_count values across the window (each has 30 non-zero minutes recorded).\n"
    "- Focus initial investigation on the frontend cluster (frontend-0/1/2) around the minute-level peaks (notably ~11:07–11:09) to determine cause of the elevated error counts."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  total_count top_cmdb_id  top_cmdb_id_count
273  2022-03-21 11:07:08 CST            6  frontend-0                  4
347  2022-03-21 11:09:00 CST            5  frontend-1                  3
290  2022-03-21 11:07:28 CST            4  frontend-0                  2
353  2022-03-21 11:09:08 CST            4  frontend-0                  2
348  2022-03-21 11:09:02 CST            4  frontend-0                  2
206  2022-03-21 11:05:18 CST            4  frontend-0                  2
987  2022-03-21 11:25:03 CST            4  frontend-0                  2
232  2022-03-21 11:05:57 CST            4  frontend-0                  2
181  2022-03-21 11:04:44 CST            4  frontend-2                  2
326  2022-03-21 11:08:18 CST            4  frontend-0                  2,                     timestamp     cmdb_id         log_name  value
522   2022-03-21 11:09:00 CST  frontend-2  log.error_count   29.0
420   2022-03-21 11:07:00 CST  frontend-2  log.error_count   27.0
316   2022-03-21 11:05:00 CST  frontend-0  log.error_count   26.0
520   2022-03-21 11:09:00 CST  frontend-1  log.error_count   25.0
644   2022-03-21 11:11:00 CST  frontend-2  log.error_count   25.0
764   2022-03-21 11:13:00 CST  frontend-2  log.error_count   24.0
270   2022-03-21 11:04:00 CST  frontend-1  log.error_count   23.0
272   2022-03-21 11:04:00 CST  frontend-2  log.error_count   23.0
418   2022-03-21 11:07:00 CST  frontend-1  log.error_count   23.0
416   2022-03-21 11:07:00 CST  frontend-0  log.error_count   23.0
1612  2022-03-21 11:27:00 CST  frontend-2  log.error_count   23.0
1490  2022-03-21 11:25:00 CST  frontend-2  log.error_count   23.0
518   2022-03-21 11:09:00 CST  frontend-0  log.error_count   23.0
1068  2022-03-21 11:18:00 CST  frontend-2  log.error_count   22.0
1370  2022-03-21 11:23:00 CST  frontend-2  log.error_count   22.0
1732  2022-03-21 11:29:00 CST  frontend-2  log.error_count   21.0
1186  2022-03-21 11:20:00 CST  frontend-1  log.error_count   21.0
1672  2022-03-21 11:28:00 CST  frontend-2  log.error_count   21.0
642   2022-03-21 11:11:00 CST  frontend-1  log.error_count   21.0
884   2022-03-21 11:15:00 CST  frontend-2  log.error_count   21.0,       cmdb_id  nonzero_count earliest_nonzero_timestamp
0  frontend-0             30    2022-03-21 19:00:00 CST
1  frontend-1             30    2022-03-21 19:00:00 CST
2  frontend-2             30    2022-03-21 19:00:00 CST)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing DataFrames if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp ints
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
log_df['timestamp'] = log_df['timestamp'].astype(int)
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
error_df['timestamp'] = error_df['timestamp'].astype(int)

# Helper: convert local Asia/Shanghai string to UTC epoch seconds
def local_to_utc_epoch(s):
    return int(pd.Timestamp(s, tz=tz).tz_convert('UTC').timestamp())

# Incident window full (for metrics computations)
window_start = "2022-03-21 11:00:00"
window_end   = "2022-03-21 11:30:00"
start_utc = local_to_utc_epoch(window_start)
end_utc   = local_to_utc_epoch(window_end)

# Minor helper to format epoch seconds to Asia/Shanghai string
def fmt_ts_s(epoch_s):
    return pd.to_datetime(int(epoch_s), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# -----------------------
# Part 1: Metrics for earliest_anomaly_timestamp == 2022-03-21 11:11:00 CST
# -----------------------

# 1.a Compute global P95 per (cmdb_id, kpi_name) using full series (global threshold)
metric_global_p95 = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Merge to full metric DF
metric_with_p95 = metric_df.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# Filter to incident window (inclusive)
metric_window = metric_with_p95[(metric_with_p95['timestamp'] >= start_utc) & (metric_with_p95['timestamp'] <= end_utc)].copy()

# Mark anomalies in window where value > global_p95
metric_window['is_anom'] = metric_window['value'] > metric_window['global_p95']

# Aggregate per group: anomaly_count, earliest_anomaly_timestamp, min and max within window
metric_anom_groups = (
    metric_window[metric_window['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anom','sum'),
        earliest_ts=('timestamp','min'),
        window_min=('value','min'),
        window_max=('value','max')
    )
)

# Attach global_p95 to the aggregated results
metric_anom_groups = metric_anom_groups.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# Format earliest timestamp to Asia/Shanghai string
metric_anom_groups['earliest_anomaly_timestamp'] = pd.to_datetime(metric_anom_groups['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_anom_groups = metric_anom_groups.drop(columns=['earliest_ts'])

# Filter to earliest_anomaly_timestamp == 2022-03-21 11:11:00 CST
target_ts_1111 = "2022-03-21 11:11:00 CST"
metric_part1 = (
    metric_anom_groups[metric_anom_groups['earliest_anomaly_timestamp'] == target_ts_1111]
    .sort_values(['anomaly_count','kpi_name'], ascending=[False, True])
    .loc[:, ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','global_p95','window_min','window_max']]
    .head(20)
    .reset_index(drop=True)
)

# -----------------------
# Part 2: Logs around 11:11 for checkoutservice-2 and shippingservice pods
# -----------------------
cmdbs_part2 = ['checkoutservice-2','shippingservice-0','shippingservice-1']
log_window_start = local_to_utc_epoch("2022-03-21 11:10:00")
log_window_end   = local_to_utc_epoch("2022-03-21 11:12:00")

logs_part2 = log_df[
    (log_df['cmdb_id'].isin(cmdbs_part2)) &
    (log_df['timestamp'] >= log_window_start) &
    (log_df['timestamp'] <= log_window_end) &
    (log_df['log_name'].isin(['log.error_count','log.row_count']))
].copy()

# Format timestamp for display
logs_part2['timestamp'] = pd.to_datetime(logs_part2['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
logs_part2 = logs_part2.sort_values('timestamp', ascending=True).loc[:, ['timestamp','cmdb_id','log_name','value']].head(20).reset_index(drop=True)

# Aggregate: non-zero log.error_count per cmdb_id in that small window
logs_error_nonzero = log_df[
    (log_df['cmdb_id'].isin(cmdbs_part2)) &
    (log_df['timestamp'] >= log_window_start) &
    (log_df['timestamp'] <= log_window_end) &
    (log_df['log_name'] == 'log.error_count') &
    (log_df['value'] > 0)
].copy()

if not logs_error_nonzero.empty:
    agg_part2 = (
        logs_error_nonzero
        .groupby('cmdb_id', as_index=False)
        .agg(nonzero_count=('value','count'), max_value=('value','max'))
        .sort_values('nonzero_count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    agg_part2 = pd.DataFrame(columns=['cmdb_id','nonzero_count','max_value'])

# -----------------------
# Part 3: Traces and logs around 11:28 for checkoutservice-2 and currencyservice-1
# -----------------------
# 3.a Compute global P95 per trace group (full series), merge, filter to window and mark anomalies
trace_global_p95 = trace_df.groupby(['cmdb_id','trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

trace_with_p95 = trace_df.merge(trace_global_p95, on=['cmdb_id','trace_name'], how='left')
trace_window = trace_with_p95[(trace_with_p95['timestamp'] >= start_utc) & (trace_with_p95['timestamp'] <= end_utc)].copy()
trace_window['is_anom'] = trace_window['value'] > trace_window['global_p95']

trace_anom_groups = (
    trace_window[trace_window['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_ts=('timestamp','min'))
)
trace_anom_groups['earliest_anomaly_timestamp'] = pd.to_datetime(trace_anom_groups['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_anom_groups = trace_anom_groups.drop(columns=['earliest_ts'])

# Filter to cmdbs and timestamp == 11:28
cmdbs_part3 = ['checkoutservice-2','currencyservice-1']
target_ts_1128 = "2022-03-21 11:28:00 CST"

trace_part3 = (
    trace_anom_groups[
        (trace_anom_groups['cmdb_id'].isin(cmdbs_part3)) &
        (trace_anom_groups['earliest_anomaly_timestamp'] == target_ts_1128)
    ]
    .sort_values('anomaly_count', ascending=False)
    .loc[:, ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp']]
    .head(20)
    .reset_index(drop=True)
)

# 3.b From log.csv filter rows for these two cmdb_ids between 11:27 and 11:29, log_name in set, order by value desc
log_window_start_327 = local_to_utc_epoch("2022-03-21 11:27:00")
log_window_end_329   = local_to_utc_epoch("2022-03-21 11:29:00")

logs_part3 = log_df[
    (log_df['cmdb_id'].isin(cmdbs_part3)) &
    (log_df['timestamp'] >= log_window_start_327) &
    (log_df['timestamp'] <= log_window_end_329) &
    (log_df['log_name'].isin(['log.error_count','log.row_count']))
].copy()

logs_part3['timestamp'] = pd.to_datetime(logs_part3['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
logs_part3 = logs_part3.sort_values('value', ascending=False).loc[:, ['timestamp','cmdb_id','log_name','value']].head(20).reset_index(drop=True)

# 3.c From error_logs.csv in 11:27–11:29 window, counts per cmdb_id and up to 5 sample messages
err_win_start = log_window_start_327
err_win_end   = log_window_end_329

error_part3 = error_df[
    (error_df['timestamp'] >= err_win_start) &
    (error_df['timestamp'] <= err_win_end) &
    (error_df['cmdb_id'].isin(cmdbs_part3))
].copy()

if not error_part3.empty:
    # group count per cmdb_id
    err_counts = error_part3.groupby('cmdb_id', as_index=False).size().rename(columns={'size':'count'})
    # sample up to 5 messages per cmdb_id (unique samples)
    samples = (
        error_part3.groupby('cmdb_id')['message']
        .apply(lambda s: list(pd.Series(s).dropna().astype(str).unique()[:5]))
        .reset_index()
        .rename(columns={'message':'sample_messages_list'})
    )
    err_merged = err_counts.merge(samples, on='cmdb_id', how='left')
    # join sample list into single string for compact display
    err_merged['sample_messages'] = err_merged['sample_messages_list'].apply(lambda lst: "; ".join(lst) if isinstance(lst, list) else "")
    error_part3_table = err_merged[['cmdb_id','count','sample_messages']].reset_index(drop=True).head(20)
else:
    error_part3_table = pd.DataFrame(columns=['cmdb_id','count','sample_messages'])

# Final compact outputs (each <=20 rows)
metric_part1, logs_part2, agg_part2, trace_part3, logs_part3, error_part3_table

```
Out[7]:
```
Summary of the three compact checks (straightforward findings and recommended focus):

1) Metrics around 11:11 (groups with earliest_anomaly_timestamp = 2022-03-21 11:11 CST)
- Many anomalous metric groups at 11:11 are container/node-related metrics (notably metrics tagged container.node-5.*) across multiple services (checkoutservice, shippingservice, productcatalogservice, adservice, emailservice, frontend, paymentservice, etc.).
- Example notable items: checkoutservice -> app.grpc.count (anomaly_count=2), shippingservice -> app.grpc.count (2), and many container.node-5 CPU/memory metrics for multiple services (each anomaly_count=2; window min/max shown in the results).
- Interpretation: a node-level resource signal (node-5) shows simultaneous anomalies across many containers at ~11:11 — this points to a likely node resource event (CPU/memory) affecting multiple services, including checkoutservice.

2) Logs for checkoutservice-2 and shippingservice-0/1 around 11:10–11:12
- Per-minute rows returned (ordered by time) show:
  - checkoutservice-2 row_count = 62 at 11:11, with log.error_count = 0 at that minute.
  - shippingservice-0/1 row_count values around 77–118 across 11:10–11:12, with log.error_count = 0.
- Aggregate of non-zero log.error_count for these three cmdb_ids in 11:10–11:12: none (empty) — no non-zero error_count recorded in that short window.
- Interpretation: at ~11:11 these services had normal/high request/row volumes but not elevated per-minute error_count; combined with node-level metric anomalies, this suggests resource pressure (node) rather than an application error spike for checkout/shipping at 11:11.

3) Traces and logs around 11:27–11:29 (checkoutservice-2 and currencyservice-1)
- Traces: two trace groups show earliest anomaly at 11:28:
  - checkoutservice-2 -> trace.to_currencyservice-1.row_count (anomaly_count=1, earliest 11:28)
  - currencyservice-1 -> trace.from_checkoutservice-2.row_count (anomaly_count=1, earliest 11:28)
- Logs (11:27–11:29):
  - currencyservice-1 has very large log.row_count values: 739 (11:27), 821 (11:28), 751 (11:29).
  - checkoutservice-2 row_count: 76 (11:27), 57 (11:28), 59 (11:29).
  - log.error_count entries for both are zero in that window.
- error_logs.csv for these two components in 11:27–11:29: none (no messages found).
- Interpretation: around 11:28 there was a pronounced traffic spike between checkoutservice-2 and currencyservice-1 (high row_count on currencyservice-1 and matching trace anomalies), but not accompanied by error log messages. This looks like a traffic/throughput surge or burst of requests, not an application error flood.

Overall conclusions and recommended next steps
- The 11:11 incident appears to be a node-level resource event (node-5) impacting many containers (multiple services show container.node-5 CPU/memory anomalies). Investigate node-5 metrics, host-level CPU/memory/disk, kubelet/container runtime and any recent scheduling/eviction or noisy-neighbor activity at ~11:11.
- The 11:28 incident is dominated by a traffic spike between checkoutservice-2 and currencyservice-1 (very high row_count on currencyservice-1 and trace anomalies). Investigate request sources, throttling/queueing at currencyservice, and any upstream retry/traffic pattern that produced the burst.
- In both windows there are few or no log.error_count non-zero spikes tied to failures and error_logs show no messages for the 11:27–11:29 pair — so focus first on resource/traffic patterns rather than application error traces.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp    global_p95    window_min    window_max
0           checkoutservice                                     app.grpc.count              2    2022-03-21 11:11:00 CST  1.755000e+01  1.800000e+01  1.800000e+01
1           shippingservice                                     app.grpc.count              2    2022-03-21 11:11:00 CST  1.216500e+02  1.230000e+02  1.240000e+02
2            emailservice-1         container.node-5.container_cpu_cfs_periods              2    2022-03-21 11:11:00 CST  5.720000e+01  5.900000e+01  6.200000e+01
3        checkoutservice2-0  container.node-5.container_cpu_cfs_throttled_s...              2    2022-03-21 11:11:00 CST  1.306896e-01  1.382812e-01  1.487299e-01
4         currencyservice-2      container.node-5.container_cpu_system_seconds              2    2022-03-21 11:11:00 CST  5.275000e-02  5.500000e-02  5.500000e-02
5               adservice-0       container.node-5.container_cpu_usage_seconds              2    2022-03-21 11:11:00 CST  2.226741e-01  2.308679e-01  2.340555e-01
6            emailservice-1       container.node-5.container_cpu_usage_seconds              2    2022-03-21 11:11:00 CST  3.262753e-02  3.304422e-02  3.607372e-02
7                frontend-1       container.node-5.container_cpu_usage_seconds              2    2022-03-21 11:11:00 CST  6.195032e-01  6.197498e-01  6.470417e-01
8          paymentservice-2       container.node-5.container_cpu_usage_seconds              2    2022-03-21 11:11:00 CST  5.243252e-03  5.276972e-03  5.933894e-03
9   productcatalogservice-2       container.node-5.container_cpu_usage_seconds              2    2022-03-21 11:11:00 CST  8.975967e-01  9.065024e-01  9.165340e-01
10              adservice-0        container.node-5.container_cpu_user_seconds              2    2022-03-21 11:11:00 CST  2.005000e-01  2.050000e-01  2.100000e-01
11  productcatalogservice-2        container.node-5.container_cpu_user_seconds              2    2022-03-21 11:11:00 CST  4.055000e-01  4.100000e-01  4.300000e-01
12              adservice-0  container.node-5.container_memory_failures.con...              2    2022-03-21 11:11:00 CST  8.560000e+01  8.650000e+01  8.700000e+01
13  productcatalogservice-2  container.node-5.container_memory_failures.con...              2    2022-03-21 11:11:00 CST  3.385700e+03  3.495500e+03  3.499500e+03
14        shippingservice-1  container.node-5.container_memory_failures.con...              2    2022-03-21 11:11:00 CST  1.262250e+02  1.440000e+02  1.570000e+02
15              adservice-0  container.node-5.container_memory_failures.hie...              2    2022-03-21 11:11:00 CST  8.560000e+01  8.650000e+01  8.700000e+01
16  productcatalogservice-2  container.node-5.container_memory_failures.hie...              2    2022-03-21 11:11:00 CST  3.385700e+03  3.495500e+03  3.499500e+03
17        shippingservice-1  container.node-5.container_memory_failures.hie...              2    2022-03-21 11:11:00 CST  1.262250e+02  1.440000e+02  1.570000e+02
18        checkoutservice-0              container.node-5.container_memory_rss              2    2022-03-21 11:11:00 CST  1.749217e+07  1.756774e+07  1.758003e+07
19        checkoutservice-0         container.node-5.container_memory_usage_MB              2    2022-03-21 11:11:00 CST  3.992441e+01  3.992969e+01  4.005078e+01,                   timestamp            cmdb_id         log_name  value
0   2022-03-21 11:10:00 CST  checkoutservice-2  log.error_count    0.0
1   2022-03-21 11:10:00 CST  checkoutservice-2    log.row_count   48.0
2   2022-03-21 11:10:00 CST  shippingservice-0  log.error_count    0.0
3   2022-03-21 11:10:00 CST  shippingservice-0    log.row_count  107.0
4   2022-03-21 11:10:00 CST  shippingservice-1  log.error_count    0.0
5   2022-03-21 11:10:00 CST  shippingservice-1    log.row_count  118.0
6   2022-03-21 11:11:00 CST  shippingservice-1    log.row_count   70.0
7   2022-03-21 11:11:00 CST  shippingservice-1  log.error_count    0.0
8   2022-03-21 11:11:00 CST  shippingservice-0    log.row_count   77.0
9   2022-03-21 11:11:00 CST  shippingservice-0  log.error_count    0.0
10  2022-03-21 11:11:00 CST  checkoutservice-2    log.row_count   62.0
11  2022-03-21 11:11:00 CST  checkoutservice-2  log.error_count    0.0
12  2022-03-21 11:12:00 CST  checkoutservice-2  log.error_count    0.0
13  2022-03-21 11:12:00 CST  checkoutservice-2    log.row_count   84.0
14  2022-03-21 11:12:00 CST  shippingservice-0  log.error_count    0.0
15  2022-03-21 11:12:00 CST  shippingservice-0    log.row_count  123.0
16  2022-03-21 11:12:00 CST  shippingservice-1  log.error_count    0.0
17  2022-03-21 11:12:00 CST  shippingservice-1    log.row_count  123.0, Empty DataFrame
Columns: [cmdb_id, nonzero_count, max_value]
Index: [],              cmdb_id                              trace_name  anomaly_count earliest_anomaly_timestamp
0  checkoutservice-2    trace.to_currencyservice-1.row_count              1    2022-03-21 11:28:00 CST
1  currencyservice-1  trace.from_checkoutservice-2.row_count              1    2022-03-21 11:28:00 CST,                   timestamp            cmdb_id         log_name  value
0   2022-03-21 11:28:00 CST  currencyservice-1    log.row_count  821.0
1   2022-03-21 11:29:00 CST  currencyservice-1    log.row_count  751.0
2   2022-03-21 11:27:00 CST  currencyservice-1    log.row_count  739.0
3   2022-03-21 11:27:00 CST  checkoutservice-2    log.row_count   76.0
4   2022-03-21 11:29:00 CST  checkoutservice-2    log.row_count   59.0
5   2022-03-21 11:28:00 CST  checkoutservice-2    log.row_count   57.0
6   2022-03-21 11:27:00 CST  currencyservice-1  log.error_count    0.0
7   2022-03-21 11:27:00 CST  checkoutservice-2  log.error_count    0.0
8   2022-03-21 11:28:00 CST  currencyservice-1  log.error_count    0.0
9   2022-03-21 11:28:00 CST  checkoutservice-2  log.error_count    0.0
10  2022-03-21 11:29:00 CST  checkoutservice-2  log.error_count    0.0
11  2022-03-21 11:29:00 CST  currencyservice-1  log.error_count    0.0, Empty DataFrame
Columns: [cmdb_id, count, sample_messages]
Index: [])```
```